<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 6 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Preserve Input, Route Rejections, and Automate Validation</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Preserve every raw input and store accepted orders and rejected records separately.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Target Doris 4.1.3 · Order data · Dedicated lab database</span>
</div>

After completion, you will have split thirteen raw input rows into ten accepted orders and three rejected records, checked rejection reasons, and tested quality rules with deliberately injected errors. Run the cells in order.

[Course notes](course6_data_quality_and_schema_validation.md) · [Course home](../README.md)


## Lab Scope

Complete the main Module 5 Lab first when possible; this section reads that historical customer table without modifying it. If only the simulated orders were loaded, the setup cell restores the missing wwi_customers table from the committed Parquet package.

Rebuild orders_raw, customers, the orders_classified view, orders_clean, and orders_rejected. The 13 input rows include three different types of errors. Stage them all before applying business acceptance rules. Module 7 reads the accepted output of this lab.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized, expected_failure
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.wwi import manifest, parquet_ddl, parquet_paths
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. Preserve the Raw Fields of Simulated New Orders

Stage amounts, order IDs, and customer IDs as strings, and identify each input with input_id. The thirteen rows include ten valid orders and one row each with an invalid amount, a missing order ID, and an invalid customer.

The customer dimension reads wwi_customers from Module 5 (663 rows), providing an independent basis for customer reference checks. Even when an input cannot be converted to the required type, its original text is preserved and the reason can be traced.


In [ ]:
lab.execute("DROP VIEW IF EXISTS orders_classified")
lab.execute("DROP TABLE IF EXISTS orders_raw")
fields = ", ".join(col + " VARCHAR(100) NULL" for col in ORDER_COLUMNS)
lab.execute(f'CREATE TABLE orders_raw (input_id BIGINT NOT NULL, {fields}) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
raw = fixture("raw_orders.json")
lab.insert("orders_raw", ["input_id", *ORDER_COLUMNS],
           [(r["input_id"], *[None if r[col] is None else str(r[col]) for col in ORDER_COLUMNS]) for r in raw])
expect(lab.query("SELECT COUNT(*) FROM orders_raw"), [(13,)])

lab.execute("DROP TABLE IF EXISTS customers")
lab.execute('CREATE TABLE customers (customer_id BIGINT NOT NULL, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
if not lab.query("SHOW TABLES LIKE 'wwi_customers'"):
    paths = parquet_paths()
    metadata = manifest()["tables"]["customers"]
    lab.execute(parquet_ddl("customers", "wwi_customers"))
    response = lab.stream_load(
        "wwi_customers", paths["customers"], "module6_wwi_" + uuid4().hex, format="parquet"
    )
    show_response(response)
    expect(response["Status"], "Success")
    expect(response["NumberLoadedRows"], metadata["rows"])
    expect(response["NumberFilteredRows"], 0)

lab.execute("INSERT INTO customers SELECT CustomerID, CustomerName FROM wwi_customers")
expect(lab.query("SELECT COUNT(*) FROM customers"), [(663,)])


## 2. Route Records Using Fixed Rules

The classification view attempts field conversions with TRY_CAST, checks the order ID, amount, customer reference, and source in sequence, and records the first matching rejection reason. Then write accepted rows to orders_clean and problematic input_id values and their reasons to orders_rejected.

Expect ten accepted rows and three rejected rows; join orders_raw on input_id to inspect the original text of each error.


In [ ]:
lab.execute("""
CREATE VIEW orders_classified AS
SELECT *, CASE
    WHEN TRY_CAST(order_id AS BIGINT) IS NULL THEN 'INVALID_ORDER_ID'
    WHEN TRY_CAST(order_amount AS DECIMAL(12,2)) IS NULL
      OR TRY_CAST(order_amount AS DECIMAL(12,2)) < 0 THEN 'INVALID_AMOUNT'
    WHEN TRY_CAST(customer_id AS BIGINT) IS NULL THEN 'INVALID_CUSTOMER'
    WHEN TRY_CAST(customer_id AS BIGINT) NOT IN (SELECT customer_id FROM customers) THEN 'INVALID_CUSTOMER'
    WHEN data_source IS NULL OR data_source <> 'COURSE_SIMULATION' THEN 'INVALID_SOURCE'
    ELSE NULL END AS reject_reason
FROM orders_raw
""")
lab.execute("DROP TABLE IF EXISTS orders_clean")
ddl = order_ddl("orders_clean")
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS orders_rejected")
lab.execute('CREATE TABLE orders_rejected (input_id BIGINT, reason VARCHAR(32)) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")
lab.execute("INSERT INTO orders_rejected SELECT input_id, reject_reason FROM orders_classified WHERE reject_reason IS NOT NULL")
lab.sql("SELECT reject_reason, COUNT(*) AS input_rows FROM orders_classified GROUP BY reject_reason ORDER BY reject_reason", title="Input routing results")
lab.sql("SELECT r.input_id, r.order_id, r.order_amount, r.customer_id, x.reason FROM orders_rejected x JOIN orders_raw r ON x.input_id=r.input_id ORDER BY r.input_id", title="Three rejected records and their raw fields")


## 3. Check Business Quality and Inject Errors

First check that all thirteen inputs have a destination, then verify order uniqueness, customer references, amounts, statuses, and complete records. Check event times against the fixed cutoff for this sample batch.

Next, try two counterexamples, calling the uniqueness check directly each time rather than waiting for aggregate checks to fail:

1. Append a duplicate order, then query and display the duplicate order ID.
2. After restoring the data, change only the second row's order ID to the first row's order ID, leaving all other fields unchanged. There are still ten rows totaling 1400.00, but one order ID is duplicated and another order is missing.

The uniqueness check must detect both errors. After each experiment, restore the accepted table from the preserved classified input and recheck all fields before handing it over to Module 7.


In [ ]:
expected_rows = order_rows(fixture("orders.json"))
def check_unique_orders():
    expect(lab.query("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders_clean"), [(0,)])

def restore_clean_orders():
    lab.execute("TRUNCATE TABLE orders_clean")
    lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")

def quality_report():
    expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")])
    check_unique_orders()
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE event_time > '2026-01-02 12:00:00' OR event_time IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE status <> 'CREATED' OR paid_amount <> 0 OR refund_amount <> 0"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE data_source <> 'COURSE_SIMULATION'"), [(0,)])
    # DATE_FORMAT makes the transport representation explicit for fixture comparison.
    projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
    expect(lab.query(f"SELECT {projection} FROM orders_clean ORDER BY order_id"), expected_rows)

expect(lab.query("SELECT input_id, reason FROM orders_rejected ORDER BY input_id"),
       [(11,"INVALID_AMOUNT"),(12,"INVALID_ORDER_ID"),(13,"INVALID_CUSTOMER")])
expect(lab.query("SELECT COUNT(*) FROM orders_classified WHERE reject_reason IS NULL"), [(10,)])
expect(lab.query("SELECT COUNT(*) FROM orders_raw r LEFT JOIN orders_rejected x ON r.input_id=x.input_id WHERE x.input_id IS NULL"), [(10,)])
quality_report()

# Trigger the uniqueness check separately so aggregate checks do not interrupt it first.
lab.insert("orders_clean", ORDER_COLUMNS, [expected_rows[0]])
lab.sql("SELECT order_id, COUNT(*) AS copies FROM orders_clean GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY order_id", title="Duplicate order ID after appending")
with expected_failure("Duplicate order check", "Duplicate order detected; the uniqueness rule works"):
    check_unique_orders()
restore_clean_orders()
quality_report()

# A subtler error: the second row uses the first row's order ID, leaving the row count and amount unchanged.
probe_rows = list(expected_rows)
probe_rows[1] = (probe_rows[0][0], *probe_rows[1][1:])
lab.execute("TRUNCATE TABLE orders_clean")
lab.insert("orders_clean", ORDER_COLUMNS, probe_rows)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")])
lab.sql("SELECT order_id, COUNT(*) AS copies FROM orders_clean GROUP BY order_id HAVING COUNT(*) > 1 ORDER BY order_id", title="Correct totals, but a duplicate order ID")
with expected_failure("Duplicate check with unchanged totals", "The row count and amount are correct, but the uniqueness check still detects the error"):
    check_unique_orders()
restore_clean_orders()
quality_report()
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_clean", title="Accepted orders restored after both error experiments")


## Completion and Your Turn

Verify thirteen input rows, ten accepted rows, and three rejected rows, and trace the three rejection reasons using input_id. Explain why customer ID 999999 can be converted to an integer but is still rejected.

Module 7 will read orders_clean and continue processing state changes for these ten accepted simulated orders.


## Independent Exercise

Add a rule: accepted orders with an amount above 200 require manual review; accept the rest. Write a SELECT with CASE to group order counts and amounts by ACCEPT and REVIEW. Expect 8 orders totaling 850.00 for ACCEPT and 2 orders totaling 550.00 for REVIEW. Try out the new rule first, leaving orders_clean unchanged for Module 7.

Write and run your code in the next cell, then expand the reference solution after completing the exercise. A blank exercise is not automatically marked as complete.


In [ ]:
# Write your SQL or load request here.


<details>
<summary>Reference solution (expand after completion)</summary>

```python
query = """SELECT CASE WHEN order_amount > 200 THEN 'REVIEW' ELSE 'ACCEPT' END AS decision,
COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_clean GROUP BY decision ORDER BY decision"""
lab.sql(query, title="New amount review rule")
expect(lab.query(query), [("ACCEPT",8,"850.00"),("REVIEW",2,"550.00")])
# Only try out the new rule; preserve the ten accepted orders used by Module 7.
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")])
```

</details>
